# 01 — Load Data

This notebook loads the raw Superstore CSV file into a SQLite database for later SQL analysis.

---

**Contents:**
1. Import libraries and define paths
2. Read the raw CSV file
3. Apply data quality fixes
4. Load the cleaned data into SQLite
5. Verify that the load was successful

---
## 1. Import libraries and define paths

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path().resolve().parent
CSV  = ROOT / "data" / "Sample - Superstore.csv"
DB   = ROOT / "data" / "superstore.db"

print(f"CSV  : {CSV}  (exists: {CSV.exists()})")
print(f"DB   : {DB}  (exists: {DB.exists()})")

---
## 2. Read the raw CSV file

In [ ]:
df = pd.read_csv(CSV, encoding="latin-1")
print(f"Rows read from CSV: {len(df):,}   Columns: {df.shape[1]}")
display(df.head(3))

---
## 3. Apply data quality fixes

Three transformations are applied before writing to SQLite. The first fix addresses a duplicate row identified during the EDA, while the second and third fixes standardize data types to make downstream SQL queries cleaner and more reliable:

### Fix 1 — Remove confirmed duplicate row

- Row IDs 3406 and 3407 were identified in the EDA as a confirmed duplicate line item with the same attributes
- Row ID 3407 is removed before loading to keep the SQLite source deduplicated

In [ ]:
before = len(df)
df = df[df["Row ID"] != 3407].reset_index(drop=True)
print(f"Dropped 1 duplicate row (Row ID 3407). Rows now: {len(df):,} (was {before:,})")

### Fix 2 — Normalize dates to ISO format (`YYYY-MM-DD`)

- `Order Date` and `Ship Date` arrive in the raw CSV as `M/D/YYYY`
- They are converted to ISO format (`YYYY-MM-DD`) to make downstream SQL date queries cleaner and easier to maintain
- This enables simpler use of SQLite date functions such as `strftime()`, `julianday()`, and `DATE()`

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y').dt.strftime('%Y-%m-%d')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y').dt.strftime('%Y-%m-%d')

### Fix 3 — Convert `Postal Code` to zero-padded text

- `Postal Code` is read as an integer by default, which silently drops the leading zero from Northeast US zip codes, e.g. `02101` → `2101`.

- Storing it as a zero-padded 5-character string preserves the correct format.

In [ ]:
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)

print(f"Order Date sample:  {df['Order Date'].iloc[0]}  (dtype: {df['Order Date'].dtype})")
print(f"Ship Date sample:   {df['Ship Date'].iloc[0]}  (dtype: {df['Ship Date'].dtype})")
print(f"Postal Code sample: {df['Postal Code'].iloc[0]}  (dtype: {df['Postal Code'].dtype})")

---
## 4. Load cleaned data into SQLite

In [ ]:
conn = sqlite3.connect(DB)
df.to_sql("orders", conn, if_exists="replace", index=False)
conn.close()
print(f"Done. {len(df):,} rows written to {DB.name} -> table 'orders'")

---
## 5. Verify database load

In [ ]:
conn = sqlite3.connect(DB)

check = pd.read_sql_query("SELECT COUNT(*) AS rows FROM orders", conn)
print("Row count:", check.to_string(index=False))

# confirm date format and postal code text type in the database
sample = pd.read_sql_query(
    'SELECT "Order Date", "Ship Date", "Postal Code" FROM orders LIMIT 3',
    conn,
)
print("\nSample of key columns:")
display(sample)

conn.close()